# M03-P01 — Neural Network Under the Hood

En M02 usamos modelos ya preparados. Ahora vamos a **abrir una red neuronal pequeña** y observar el aprendizaje paso a paso.

Queremos ver de forma tangible:

```text
entrada → forward → logit/probabilidad → loss → backward → gradientes → optimizer.step()
```

Usaremos un problema 2D pequeño para poder visualizar los datos y la frontera de decisión.

> Esta práctica no usa AWS ni Bedrock. Todo se ejecuta en CPU dentro del Codespace.

## 1. Preparar el laboratorio

Usaremos NumPy, pandas, scikit-learn, PyTorch y matplotlib. Fijamos una semilla para que los resultados sean reproducibles.

In [ ]:
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
DEVICE = torch.device('cpu')

print('PyTorch:', torch.__version__)
print('Device:', DEVICE)

## 2. Un problema que podamos ver

Generaremos dos grupos de puntos con forma de medias lunas. Cada punto tiene dos características, `x1` y `x2`, y una etiqueta binaria `0` o `1`. La frontera no es perfectamente lineal.

In [ ]:
X, y = make_moons(n_samples=600, noise=0.23, random_state=SEED)

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.30, stratify=y, random_state=SEED
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train).astype('float32')
X_val_scaled = scaler.transform(X_val).astype('float32')
y_train_float = y_train.astype('float32')
y_val_float = y_val.astype('float32')

print('Train:', X_train_scaled.shape, y_train_float.shape)
print('Validation:', X_val_scaled.shape, y_val_float.shape)
print('Clase 0 / 1 en train:', np.bincount(y_train))

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(X_train_scaled[:, 0], X_train_scaled[:, 1], c=y_train, alpha=0.75)
ax.set_title('Datos de entrenamiento')
ax.set_xlabel('x1 estandarizada')
ax.set_ylabel('x2 estandarizada')
ax.grid(alpha=0.2)
plt.show()

Hay estructura, ruido y zonas próximas a la frontera. Eso nos permitirá observar ejemplos fáciles, ejemplos inciertos, errores y sobreajuste.

## 3. De arrays a tensores

PyTorch trabaja con tensores. Aquí usamos `float32` y CPU.

In [ ]:
X_train_t = torch.tensor(X_train_scaled, dtype=torch.float32, device=DEVICE)
y_train_t = torch.tensor(y_train_float, dtype=torch.float32, device=DEVICE)
X_val_t = torch.tensor(X_val_scaled, dtype=torch.float32, device=DEVICE)
y_val_t = torch.tensor(y_val_float, dtype=torch.float32, device=DEVICE)

print('X_train:', X_train_t.shape, X_train_t.dtype)
print('y_train:', y_train_t.shape, y_train_t.dtype)
print('Primer punto:', X_train_t[0])
print('Su etiqueta:', y_train_t[0])

## 4. Construir una red neuronal mínima

Arquitectura:

```text
2 entradas → Linear(2→8) → ReLU → Linear(8→1) → logit
```

El `logit` todavía no es una probabilidad; aplicaremos `sigmoid` cuando queramos interpretarlo como `P(clase 1)`.

In [ ]:
class TinyMLP(nn.Module):
    def __init__(self, hidden_units=8):
        super().__init__()
        self.hidden = nn.Linear(2, hidden_units)
        self.activation = nn.ReLU()
        self.output = nn.Linear(hidden_units, 1)

    def forward(self, x):
        z = self.hidden(x)
        h = self.activation(z)
        return self.output(h).squeeze(1)

torch.manual_seed(SEED)
model = TinyMLP(hidden_units=8).to(DEVICE)
print(model)

rows = []
for name, parameter in model.named_parameters():
    rows.append({
        'parameter': name,
        'shape': tuple(parameter.shape),
        'values': parameter.numel(),
        'requires_grad': parameter.requires_grad,
    })
display(pd.DataFrame(rows))
print('Parámetros entrenables totales:', sum(p.numel() for p in model.parameters()))

Con 8 neuronas ocultas tenemos 33 parámetros: `2×8 + 8 = 24` en la primera capa y `8×1 + 1 = 9` en la salida. Estos son los números que el entrenamiento modificará.

## 5. Abrir un `forward pass`

Pasaremos un solo punto por cada etapa: entrada → capa lineal → ReLU → logit → sigmoid.

In [ ]:
model.eval()
x_example = X_train_t[0:1]
y_example = y_train_t[0].item()

with torch.no_grad():
    z1 = model.hidden(x_example)
    h1 = model.activation(z1)
    logit = model.output(h1).squeeze(1)
    probability = torch.sigmoid(logit)

print('Entrada:', x_example.squeeze(0).tolist())
print('Etiqueta real:', int(y_example))
print('Salida lineal z1:', z1.squeeze(0))
print('Después de ReLU h1:', h1.squeeze(0))
print('Logit:', round(logit.item(), 4))
print('Probabilidad clase 1:', round(probability.item(), 4))
print('Predicción con umbral 0.5:', int(probability.item() >= 0.5))

`ReLU(x)` convierte los valores negativos en cero y mantiene los positivos. Esta no linealidad permite a la red aprender fronteras más complejas que una sola transformación lineal.

## 6. La `loss`

Usaremos `BCEWithLogitsLoss`, que combina de forma estable sigmoid y binary cross entropy. La loss no es accuracy: proporciona una señal continua que el optimizador puede minimizar.

In [ ]:
loss_fn = nn.BCEWithLogitsLoss()
small_X = X_train_t[:8]
small_y = y_train_t[:8]

model.train()
small_logits = model(small_X)
small_loss = loss_fn(small_logits, small_y)
small_probs = torch.sigmoid(small_logits)

view = pd.DataFrame({
    'target': small_y.detach().cpu().numpy().astype(int),
    'logit': small_logits.detach().cpu().numpy(),
    'prob_class_1': small_probs.detach().cpu().numpy(),
    'prediction': (small_probs >= 0.5).detach().cpu().numpy().astype(int),
})
display(view.round(4))
print('Loss del pequeño lote:', round(small_loss.item(), 6))

## 7. `backward()`: calcular gradientes

`loss.backward()` calcula cómo cambia la loss respecto a cada parámetro. Todavía **no cambia los pesos**. El resultado queda en `.grad`.

In [ ]:
model.zero_grad()
small_logits = model(small_X)
small_loss = loss_fn(small_logits, small_y)
small_loss.backward()

rows = []
for name, parameter in model.named_parameters():
    rows.append({
        'parameter': name,
        'shape': tuple(parameter.shape),
        'grad_norm': parameter.grad.norm().item(),
        'grad_abs_mean': parameter.grad.abs().mean().item(),
        'first_grad_value': parameter.grad.flatten()[0].item(),
    })
display(pd.DataFrame(rows).round(6))

La secuencia hasta aquí es `forward → loss → backward → gradientes`. Falta el paso que realmente actualiza los parámetros.

## 8. `optimizer.step()`: modificar los pesos

Usamos una sola actualización SGD para comprobar que `peso nuevo ≈ peso anterior - learning_rate × gradiente`.

In [ ]:
torch.manual_seed(SEED)
step_model = TinyMLP(hidden_units=8).to(DEVICE)
step_optimizer = torch.optim.SGD(step_model.parameters(), lr=0.10)

before = step_model.hidden.weight.detach().clone()
step_optimizer.zero_grad()
step_logits = step_model(small_X)
step_loss = loss_fn(step_logits, small_y)
step_loss.backward()

first_gradient = step_model.hidden.weight.grad[0, 0].item()
first_before = before[0, 0].item()
step_optimizer.step()
first_after = step_model.hidden.weight.detach()[0, 0].item()

print('Primer peso antes:      ', round(first_before, 6))
print('Su gradiente:           ', round(first_gradient, 6))
print('Learning rate:          ', step_optimizer.param_groups[0]['lr'])
print('Primer peso después:    ', round(first_after, 6))
print('Cambio real:            ', round(first_after - first_before, 6))
print('Cambio esperado aprox.: ', round(-0.10 * first_gradient, 6))

Una época de entrenamiento repite: `forward → loss → zero_grad → backward → optimizer.step`. En datasets grandes normalmente se hace por mini-batches.

## 9. Entrenamiento completo

Entrenaremos con Adam y registraremos train/validation loss, accuracy y norma global del gradiente.

In [ ]:
def accuracy_from_logits(logits, targets):
    predictions = (torch.sigmoid(logits) >= 0.5).float()
    return (predictions == targets).float().mean().item()

def gradient_global_norm(network):
    squared = 0.0
    for parameter in network.parameters():
        if parameter.grad is not None:
            squared += parameter.grad.detach().norm().item() ** 2
    return math.sqrt(squared)

def train_network(hidden_units=8, learning_rate=0.03, epochs=200, seed=SEED):
    torch.manual_seed(seed)
    network = TinyMLP(hidden_units=hidden_units).to(DEVICE)
    optimizer = torch.optim.Adam(network.parameters(), lr=learning_rate)
    history = []

    for epoch in range(epochs + 1):
        network.train()
        optimizer.zero_grad()
        train_logits = network(X_train_t)
        train_loss = loss_fn(train_logits, y_train_t)
        train_loss.backward()
        grad_norm = gradient_global_norm(network)
        optimizer.step()

        network.eval()
        with torch.no_grad():
            current_train_logits = network(X_train_t)
            val_logits = network(X_val_t)
            current_train_loss = loss_fn(current_train_logits, y_train_t).item()
            val_loss = loss_fn(val_logits, y_val_t).item()
            train_acc = accuracy_from_logits(current_train_logits, y_train_t)
            val_acc = accuracy_from_logits(val_logits, y_val_t)

        history.append({
            'epoch': epoch,
            'train_loss': current_train_loss,
            'val_loss': val_loss,
            'train_accuracy': train_acc,
            'val_accuracy': val_acc,
            'grad_norm': grad_norm,
        })

    return network, pd.DataFrame(history)

model, history = train_network(hidden_units=8, learning_rate=0.03, epochs=200)
display(history.iloc[[0, 1, 5, 20, 50, 100, 200]].round(4))

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(history['epoch'], history['train_loss'], label='train')
ax.plot(history['epoch'], history['val_loss'], label='validation')
ax.set_title('Loss durante el entrenamiento')
ax.set_xlabel('Epoch')
ax.set_ylabel('Loss')
ax.legend()
ax.grid(alpha=0.2)
plt.show()

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(history['epoch'], history['train_accuracy'], label='train')
ax.plot(history['epoch'], history['val_accuracy'], label='validation')
ax.set_title('Accuracy durante el entrenamiento')
ax.set_xlabel('Epoch')
ax.set_ylabel('Accuracy')
ax.set_ylim(0.45, 1.01)
ax.legend()
ax.grid(alpha=0.2)
plt.show()

print('Accuracy final train:', f"{history.iloc[-1]['train_accuracy']:.1%}")
print('Accuracy final validation:', f"{history.iloc[-1]['val_accuracy']:.1%}")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(history['epoch'], history['grad_norm'])
ax.set_title('Norma global del gradiente')
ax.set_xlabel('Epoch')
ax.set_ylabel('Gradient norm')
ax.grid(alpha=0.2)
plt.show()

Un entrenamiento saludable suele mostrar train y validation loss descendiendo. Validation representa datos que no se usan para actualizar los pesos.

## 10. Ver la frontera aprendida

Como solo tenemos dos características, podemos evaluar una rejilla y visualizar `P(clase 1)`.

In [ ]:
def plot_decision_boundary(network, X_points, y_points, title):
    x_min, x_max = X_points[:, 0].min() - 0.7, X_points[:, 0].max() + 0.7
    y_min, y_max = X_points[:, 1].min() - 0.7, X_points[:, 1].max() + 0.7
    xx, yy = np.meshgrid(
        np.linspace(x_min, x_max, 250),
        np.linspace(y_min, y_max, 250),
    )
    grid = np.c_[xx.ravel(), yy.ravel()].astype('float32')
    grid_t = torch.tensor(grid, dtype=torch.float32, device=DEVICE)

    network.eval()
    with torch.no_grad():
        probabilities = torch.sigmoid(network(grid_t)).cpu().numpy()

    zz = probabilities.reshape(xx.shape)
    fig, ax = plt.subplots(figsize=(7, 5.5))
    contour = ax.contourf(xx, yy, zz, levels=np.linspace(0, 1, 11), alpha=0.45)
    ax.contour(xx, yy, zz, levels=[0.5], linewidths=2)
    ax.scatter(X_points[:, 0], X_points[:, 1], c=y_points, edgecolors='black', linewidths=0.3)
    ax.set_title(title)
    ax.set_xlabel('x1')
    ax.set_ylabel('x2')
    fig.colorbar(contour, ax=ax, label='P(clase 1)')
    plt.show()

plot_decision_boundary(model, X_val_scaled, y_val, 'Frontera de decisión — modelo entrenado')

La línea de `P(clase 1)=0.5` es la frontera donde cambia la clase predicha.

## 11. El `learning rate` importa

Compararemos `0.0001`, `0.03` y `10.0` con la misma arquitectura e inicialización. Más grande no significa necesariamente más rápido.

In [ ]:
lr_rows = []
lr_curves = []

for lr in (0.0001, 0.03, 10.0):
    network, run_history = train_network(
        hidden_units=8, learning_rate=lr, epochs=80, seed=SEED
    )
    last = run_history.iloc[-1]
    lr_rows.append({
        'learning_rate': lr,
        'final_train_loss': last['train_loss'],
        'final_val_loss': last['val_loss'],
        'final_train_accuracy': last['train_accuracy'],
        'final_val_accuracy': last['val_accuracy'],
    })
    lr_curves.append(run_history[['epoch', 'train_loss']].assign(learning_rate=lr))

lr_results = pd.DataFrame(lr_rows)
lr_curves = pd.concat(lr_curves, ignore_index=True)
display(lr_results.style.format({
    'final_train_loss': '{:.4f}',
    'final_val_loss': '{:.4f}',
    'final_train_accuracy': '{:.1%}',
    'final_val_accuracy': '{:.1%}',
}))

fig, ax = plt.subplots(figsize=(8, 4.5))
for lr, group in lr_curves.groupby('learning_rate'):
    ax.plot(group['epoch'], group['train_loss'], label=f'lr={lr}')
ax.set_title('Efecto del learning rate')
ax.set_xlabel('Epoch')
ax.set_ylabel('Train loss')
ax.legend()
ax.grid(alpha=0.2)
plt.show()

Un learning rate demasiado pequeño aprende lentamente; uno razonable converge; uno excesivo puede impedir el aprendizaje.

## 12. Capacidad: número de neuronas ocultas

Comparamos 2, 8 y 64 neuronas. Más parámetros permiten representar funciones más complejas, pero no garantizan mejor generalización.

In [ ]:
capacity_rows = []
for hidden_units in (2, 8, 64):
    network, run_history = train_network(
        hidden_units=hidden_units, learning_rate=0.03, epochs=150, seed=SEED
    )
    last = run_history.iloc[-1]
    capacity_rows.append({
        'hidden_units': hidden_units,
        'parameters': sum(p.numel() for p in network.parameters()),
        'train_accuracy': last['train_accuracy'],
        'val_accuracy': last['val_accuracy'],
        'train_loss': last['train_loss'],
        'val_loss': last['val_loss'],
    })

capacity_df = pd.DataFrame(capacity_rows)
display(capacity_df.style.format({
    'train_accuracy': '{:.1%}',
    'val_accuracy': '{:.1%}',
    'train_loss': '{:.4f}',
    'val_loss': '{:.4f}',
}))

## 13. Provocar sobreajuste para poder verlo

Creamos deliberadamente una situación adversa: solo 80 ejemplos de entrenamiento, una red grande y 25 % de etiquetas alteradas. Validation permanece limpia. El objetivo es hacer visible la memorización de ruido.

In [ ]:
rng = np.random.default_rng(SEED)
tiny_indices = rng.choice(len(X_train_scaled), size=80, replace=False)
X_tiny_np = X_train_scaled[tiny_indices].copy()
y_tiny_np = y_train_float[tiny_indices].copy()
flip_indices = rng.choice(len(y_tiny_np), size=20, replace=False)
y_tiny_noisy_np = y_tiny_np.copy()
y_tiny_noisy_np[flip_indices] = 1 - y_tiny_noisy_np[flip_indices]

X_tiny_t = torch.tensor(X_tiny_np, dtype=torch.float32)
y_tiny_noisy_t = torch.tensor(y_tiny_noisy_np, dtype=torch.float32)

print('Ejemplos tiny train:', len(X_tiny_t))
print('Etiquetas alteradas:', len(flip_indices))
print('Porcentaje de ruido:', f"{len(flip_indices) / len(X_tiny_t):.0%}")

In [ ]:
class BiggerMLP(nn.Module):
    def __init__(self, hidden_units=64):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(2, hidden_units), nn.ReLU(),
            nn.Linear(hidden_units, hidden_units), nn.ReLU(),
            nn.Linear(hidden_units, 1),
        )

    def forward(self, x):
        return self.network(x).squeeze(1)

torch.manual_seed(SEED)
overfit_model = BiggerMLP(hidden_units=64).to(DEVICE)
optimizer = torch.optim.Adam(overfit_model.parameters(), lr=0.01)
overfit_rows = []

for epoch in range(801):
    overfit_model.train()
    optimizer.zero_grad()
    logits = overfit_model(X_tiny_t)
    train_loss = loss_fn(logits, y_tiny_noisy_t)
    train_loss.backward()
    optimizer.step()

    if epoch % 10 == 0:
        overfit_model.eval()
        with torch.no_grad():
            train_logits = overfit_model(X_tiny_t)
            val_logits = overfit_model(X_val_t)
            overfit_rows.append({
                'epoch': epoch,
                'train_loss': loss_fn(train_logits, y_tiny_noisy_t).item(),
                'val_loss': loss_fn(val_logits, y_val_t).item(),
                'train_accuracy': accuracy_from_logits(train_logits, y_tiny_noisy_t),
                'val_accuracy': accuracy_from_logits(val_logits, y_val_t),
            })

overfit_history = pd.DataFrame(overfit_rows)
best = overfit_history.loc[overfit_history['val_loss'].idxmin()]
final = overfit_history.iloc[-1]
print('Mejor validation loss:')
display(best.to_frame().T.round(4))
print('Final del entrenamiento:')
display(final.to_frame().T.round(4))

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(overfit_history['epoch'], overfit_history['train_loss'], label='train con ruido')
ax.plot(overfit_history['epoch'], overfit_history['val_loss'], label='validation limpia')
ax.set_title('Sobreajuste: train mejora mientras validation empeora')
ax.set_xlabel('Epoch')
ax.set_ylabel('Loss')
ax.legend()
ax.grid(alpha=0.2)
plt.show()

plot_decision_boundary(
    overfit_model, X_tiny_np, y_tiny_noisy_np.astype(int),
    'Frontera de una red grande tras memorizar ruido'
)

Sobreajustar significa que el modelo puede llegar a memorizar train —incluido ruido— mientras validation deja de mejorar o empeora. Train perfecto no implica buena generalización.

## 14. Mirar predicciones inciertas

Una probabilidad cercana a 0.5 indica que, para este modelo, el punto está cerca de la frontera de decisión.

In [ ]:
model.eval()
with torch.no_grad():
    val_logits = model(X_val_t)
    val_probabilities = torch.sigmoid(val_logits).cpu().numpy()

uncertainty = np.abs(val_probabilities - 0.5)
indices = np.argsort(uncertainty)[:10]
uncertain_df = pd.DataFrame({
    'x1': X_val_scaled[indices, 0],
    'x2': X_val_scaled[indices, 1],
    'true_label': y_val[indices],
    'prob_class_1': val_probabilities[indices],
    'prediction': (val_probabilities[indices] >= 0.5).astype(int),
    'distance_to_0.5': uncertainty[indices],
})
display(uncertain_df.round(4))

## 15. Qué puedes modificar

Prueba cambios controlados:

- learning rate: `0.001`, `0.01`, `0.1`, `1.0`;
- hidden units: `4`, `16`, `32`, `128`;
- más o menos epochs.

Pregunta siempre: ¿mejora train?, ¿mejora validation?, ¿aumenta el gap?, ¿cambia la frontera?

## 16. Qué debes llevarte

Una red pequeña ya contiene los ingredientes conceptuales de redes mucho mayores: parámetros, forward, activaciones, logits, loss, gradientes, backpropagation, optimizador, epochs, generalización y sobreajuste.

## 17. Preguntas de reflexión

1. ¿Qué es un parámetro entrenable?
2. ¿Por qué la red tiene 33 parámetros?
3. ¿Qué diferencia hay entre logit y probabilidad?
4. ¿Qué aporta ReLU?
5. ¿Qué mide la loss?
6. ¿Por qué loss y accuracy no son lo mismo?
7. ¿Qué hace `backward()`?
8. ¿`backward()` cambia los pesos?
9. ¿Qué hace `optimizer.step()`?
10. ¿Por qué usamos `zero_grad()`?
11. ¿Qué ocurre con un learning rate demasiado pequeño o demasiado grande?
12. ¿Más neuronas garantizan mejor generalización?
13. ¿Cómo reconocemos sobreajuste?
14. ¿Por qué necesitamos validation?
15. ¿Qué significa una probabilidad cercana a 0.5?

## 18. Puente hacia M03-P02

En la siguiente práctica abriremos otro mecanismo fundamental:

```text
tokens → embeddings → Q/K/V → attention scores → representación contextual
```

No construiremos un Transformer grande desde cero; queremos **ver** cómo funciona la atención.